# Solutions: Refactor Monolithic Code to Modular Code (Imputation Example)

This notebook provides step-by-step solutions for refactoring a monolithic imputation script into modular code, following RAP best practices. Each step matches the corresponding exercise notebook.

## Step 1 Solution: Wrap Imputation in a Function

Below is a function that performs imputation for missing height and weight values using column means.

In [ ]:
import pandas as pd


def impute_height_weight(df: pd.DataFrame) -> pd.DataFrame:
    """
    Impute missing height and weight with column means.
    Args:
        df (pd.DataFrame): Input DataFrame.
    Returns:
        pd.DataFrame: DataFrame with imputed values.
    """
    df = df.copy()
    df["height_cm"] = df["height_cm"].fillna(df["height_cm"].mean())
    df["weight_kg"] = df["weight_kg"].fillna(df["weight_kg"].mean())
    return df


# Example usage:
df = pd.read_csv("../data/input/health_data.csv")
cleaned = df.dropna(subset=["diagnosis"])
imputed = impute_height_weight(cleaned)
print(imputed.head())

## Step 2 Solution: Add Function to Pipeline

You would place the `impute_height_weight` function in `src/python_rap_demo/processing.py` and import it in your main pipeline script. Here is how you would use it in the pipeline:

In [ ]:
from python_rap_demo.cleaning import drop_missing_diagnosis
from python_rap_demo.io import read_health_data
from python_rap_demo.processing import impute_height_weight

# Load and clean data
df = read_health_data("../data/input/health_data.csv")
cleaned = drop_missing_diagnosis(df)
imputed = impute_height_weight(cleaned)
print(imputed.head())

## Step 3 Solution: Reflection

- Modular code makes it easier to test, maintain, and extend your pipeline.
- Functions can be reused and unit tested independently.
- Separating code into modules clarifies the workflow and supports reproducibility.

**Example extension:**
You could add a function to impute missing age values, or to handle categorical variables differently.

## Step 4 Solution (Harder): Refactor Visualisation Code

Below are example functions for visualising missing values and disease prevalence using Plotly. These would go in `src/python_rap_demo/report.py`.

In [ ]:
import plotly.express as px


def plot_missing_values(df: pd.DataFrame, output_path: str) -> None:
    """
    Plot number of missing values per column and save as PNG.
    """
    missing_counts = df.isnull().sum().reset_index()
    missing_counts.columns = ["column", "missing_count"]
    fig = px.bar(
        missing_counts,
        x="column",
        y="missing_count",
        title="Number of Missing Values per Column",
        labels={"missing_count": "Missing Count", "column": "Column"},
        color_discrete_sequence=["#C44E52"],
    )
    fig.write_image(output_path)


def plot_disease_prevalence(df: pd.DataFrame, output_path: str) -> None:
    """
    Plot disease prevalence as a bar chart and save as PNG.
    """
    prevalence = df["diagnosis"].value_counts(normalize=True).reset_index()
    prevalence.columns = ["diagnosis", "proportion"]
    fig = px.bar(
        prevalence,
        x="diagnosis",
        y="proportion",
        color="diagnosis",
        title="Disease Prevalence",
        labels={"proportion": "Proportion", "diagnosis": "Diagnosis"},
        color_discrete_sequence=["#4C72B0", "#55A868", "#C44E52"],
    )
    fig.write_image(output_path)


# Example usage:
plot_missing_values(df, "../exercises/outputs/missing_values.png")
plot_disease_prevalence(imputed, "../exercises/outputs/prevalence.png")